# Gold Standard Sampling
Fetching judgments outside LIDO for manual annotation, from Rechtspraak API.

**Required files in the same directory:**
- `eclis.csv.gz` — all ECLIs from rechtspraak.nl
- `links.csv.gz` — LIDO annotations

**Output:**
- `gold_standard_sample.csv.gz` — 1000 judgments with text


In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import random
import time

CONTENT_URL = "https://data.rechtspraak.nl/uitspraken/content"
SEARCH_URL  = "https://data.rechtspraak.nl/uitspraken/zoeken"
NS = {"atom": "http://www.w3.org/2005/Atom"}

# Creating sample from local dataset (not necessary when sample csv available)

In [2]:
# # LIDO ECLIs
# df_links = pd.read_csv("data/links.csv.gz", compression="gzip", usecols=["ecli"])
# lido_eclis = set(df_links["ecli"].dropna().unique())
# print(f"ECLIs in LIDO: {len(lido_eclis):,}")

# # Rechtspraak.nl ECLIs
# df_eclis = pd.read_csv("data/eclis.csv.gz", compression="gzip")
# print(f"ECLIs on rechtspraak.nl: {len(df_eclis):,}")

# candidates = df_eclis[~df_eclis["ecli"].isin(lido_eclis)].copy()
# candidates_set = set(candidates["ecli"].tolist())

# print(f"Candidates outside LIDO:      {len(candidates):,}")


In [3]:
# random.seed(42)
# N_TOTAL = 1000

# sample_eclis = random.sample(list(candidates_set), N_TOTAL)
# df_sample = pd.DataFrame({"ecli": sample_eclis})

# print(f"Randomly sampled: {len(df_sample)} judgments")
# df_sample.head()


In [4]:
# # Save sample permanently
# df_sample.to_csv("gold_standard_sample.csv.gz", index=False, compression="gzip")
# print(f"Saved: {len(df_sample)} judgments")

# Load sample csv

In [5]:
df_sample = pd.read_csv("gold_standard_sample.csv.gz", compression="gzip")
print(f"Loaded: {len(df_sample)} judgments")

Loaded: 1000 judgments


In [6]:
def fetch_text(ecli):
    """Fetches plain text, date, and rechtsgebied via the content API."""
    try:
        r = requests.get(
            f"https://data.rechtspraak.nl/uitspraken/content?id={ecli}",
            timeout=10
        )

        if r.status_code == 429:
            print("  Rate limit hit, waiting 30s...")
            time.sleep(30)
            r = requests.get(
                f"https://data.rechtspraak.nl/uitspraken/content?id={ecli}",
                timeout=10
            )

        if r.status_code != 200:
            return None, None, None

        root = ET.fromstring(r.content)
        ns = {"rs": "http://www.rechtspraak.nl/schema/rechtspraak-1.0"}

        # Judgment text — only the uitspraak/conclusie body
        def get_text(element):
            text = element.text or ""
            for child in element:
                text += get_text(child)
                text += child.tail or ""
            return text

        uitspraak = root.find(".//rs:uitspraak", ns)
        if uitspraak is None:
            uitspraak = root.find(".//rs:conclusie", ns)

        if uitspraak is None or len(get_text(uitspraak).strip()) < 200:
            return None, None, None

        text = get_text(uitspraak).strip()

        # Date
        date = None
        for tag in ["{http://purl.org/dc/terms/}date",
                    "{http://www.rechtspraak.nl/schema/rechtspraak-1.0}date"]:
            elem = root.find(f".//{tag}")
            if elem is not None and elem.text:
                date = elem.text.strip()
                break

        # Rechtsgebied
        rechtsgebied = None
        for tag in ["{http://purl.org/dc/terms/}subject",
                    "{http://www.rechtspraak.nl/schema/rechtspraak-1.0}subject"]:
            elem = root.find(f".//{tag}")
            if elem is not None and elem.text:
                rechtsgebied = elem.text.strip()
                break

        return text, date, rechtsgebied

    except Exception:
        return None, None, None

In [ ]:
texts, dates, rechtsgebieden = [], [], []
failed = 0

for i, row in df_sample.iterrows():
    text, date, rechtsgebied = fetch_text(row["ecli"])
    texts.append(text)
    dates.append(date)
    rechtsgebieden.append(rechtsgebied)

    if text is None:
        failed += 1

    if (i + 1) % 20 == 0:
        print(f"{i + 1}/{len(df_sample)} fetched ({failed} failed)")

    time.sleep(0.3)

df_sample["text"]             = texts
df_sample["date"]             = dates
df_sample["rechtsgebied"]     = rechtsgebieden

df_sample = df_sample[df_sample["text"].notna()].reset_index(drop=True)
print(f"\nRemaining after filtering: {len(df_sample)}")
df_sample[["ecli", "date", "rechtsgebied"]].head(10)

20/1000 fetched (0 failed)
40/1000 fetched (0 failed)
60/1000 fetched (0 failed)
80/1000 fetched (0 failed)
100/1000 fetched (0 failed)
120/1000 fetched (0 failed)
140/1000 fetched (0 failed)
160/1000 fetched (0 failed)
180/1000 fetched (0 failed)
200/1000 fetched (0 failed)
220/1000 fetched (0 failed)
240/1000 fetched (0 failed)
260/1000 fetched (0 failed)
280/1000 fetched (0 failed)
300/1000 fetched (0 failed)
320/1000 fetched (0 failed)
340/1000 fetched (0 failed)
360/1000 fetched (0 failed)
380/1000 fetched (0 failed)
400/1000 fetched (0 failed)
420/1000 fetched (0 failed)
440/1000 fetched (0 failed)
460/1000 fetched (0 failed)
480/1000 fetched (0 failed)
500/1000 fetched (0 failed)
520/1000 fetched (0 failed)
540/1000 fetched (0 failed)
560/1000 fetched (0 failed)
580/1000 fetched (0 failed)
600/1000 fetched (0 failed)
620/1000 fetched (0 failed)
640/1000 fetched (0 failed)
660/1000 fetched (0 failed)
680/1000 fetched (0 failed)
700/1000 fetched (0 failed)
720/1000 fetched (0 fail

# Small EDA

In [ ]:
df_sample.head(10)

,ecli,text,date,rechtsgebied
0,ECLI:NL:RBDHA:2025:8235,RECHTBANK DEN HAAG\n \n \n Bestuursrecht\...,2025-04-10,Bestuursrecht; Vreemdelingenrecht
1,ECLI:NL:OGAACMB:2017:68,Uitspraak van 28 augustus 2017\t\t\t\t\t\n ...,2017-08-28,Bestuursrecht; Ambtenarenrecht
2,ECLI:NL:RBSGR:2012:BW2477,RECHTBANK 'S-GRAVENHAGE\n \n Sector civi...,2012-04-16,Civiel recht
3,ECLI:NL:RBDHA:2021:16863,RECHTBANK DEN HAAG\n \n \n Zittings...,2021-11-23,Bestuursrecht; Vreemdelingenrecht
4,ECLI:NL:RBDHA:2025:4857,RECHTBANK DEN HAAG\n \n \n Zittings...,2025-03-25,Bestuursrecht; Vreemdelingenrecht
5,ECLI:NL:RBNNE:2024:4711,RECHTBANK NOORD-NEDERLAND\n \n \n Zitting...,2024-11-14,Bestuursrecht; Belastingrecht
6,ECLI:NL:RVS:2025:2901,BRS.25.000062\n ECLI:NL:RVS:2025:2901\n ...,2025-07-01,Bestuursrecht; Vreemdelingenrecht
7,ECLI:NL:RBARN:2006:3841,vonni\n s\n \n \n \n RE...,2006-06-21,Civiel recht
8,ECLI:NL:RBDHA:2025:13255,RECHTBANK DEN HAAG\n \n \n Zittings...,2025-07-18,Bestuursrecht; Vreemdelingenrecht
9,ECLI:NL:CBB:2025:206,proces-verbaal uitspraak \n \n \n \n ...,2025-02-27,Bestuursrecht


In [ ]:
print("=== Distribution by rechtsgebied ===")
print(df_sample["rechtsgebied"].value_counts(dropna=False).to_string())

print("\n=== Date range ===")
print(f"Earliest: {df_sample['date'].min()}")
print(f"Latest:   {df_sample['date'].max()}")

print("\n=== Text length (characters) ===")
print(df_sample["text"].str.len().describe().round(0))

=== Distribution by rechtsgebied ===
rechtsgebied
Civiel recht                                  192
Strafrecht                                    171
Bestuursrecht; Vreemdelingenrecht             145
Bestuursrecht                                 131
Civiel recht; Personen- en familierecht        89
Bestuursrecht; Belastingrecht                  69
Civiel recht; Verbintenissenrecht              55
Bestuursrecht; Socialezekerheidsrecht          34
Bestuursrecht; Bestuursprocesrecht             22
Bestuursrecht; Omgevingsrecht                  19
Bestuursrecht; Ambtenarenrecht                 10
Civiel recht; Arbeidsrecht                     10
Civiel recht; Ondernemingsrecht                10
Civiel recht; Insolventierecht                 10
Civiel recht; Burgerlijk procesrecht            9
Strafrecht; Europees strafrecht                 7
Strafrecht; Materieel strafrecht                6
Strafrecht; Strafprocesrecht                    5
Bestuursrecht; Bestuursstrafrecht               5


In [ ]:
df_sample["text"].str.len().describe()

count      1000.000000
mean      15714.258000
std       21912.980616
min         297.000000
25%        6439.750000
50%       10557.500000
75%       17599.500000
max      454585.000000
Name: text, dtype: float64

In [ ]:
# Bekijk de verdeling beter
print("Uitspraken per lengtecat:")
bins = [0, 10000, 30000, 60000, float("inf")]
labels = ["kort (<10k)", "middel (10-30k)", "lang (30-60k)", "zeer lang (>60k)"]
df_sample["lengte_cat"] = pd.cut(df_sample["text"].str.len(), bins=bins, labels=labels)
print(df_sample["lengte_cat"].value_counts().sort_index())

Uitspraken per lengtecat:
lengte_cat
kort (<10k)         470
middel (10-30k)     428
lang (30-60k)        79
zeer lang (>60k)     23
Name: count, dtype: int64


# Annotatie Anthropic Haiku


In [ ]:
import anthropic
import json
import time
import os
from dotenv import load_dotenv
load_dotenv()

# Haal de API key op
claude_key = os.getenv("claude_key")

os.environ["ANTHROPIC_API_KEY"] = claude_key
client = anthropic.Anthropic()

# ── Prompts ───────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are an expert annotator of Dutch legal texts. Your task is to identify all legal references in a court judgment.

You must identify references of exactly four types:
- BWB: References to Dutch legislation — laws, articles, paragraphs, subsections. Includes full names, abbreviations (Awb, BW, Sr, Sv, Rv, Gw etc.) and article references like "art. 6:162 BW" or "artikel 8 lid 1 Awb".
- ECLI: References to court decisions — via ECLI number, court name + date, case name (roepnaam), or journal citation like "NJ 2020/123".
- CELEX: References to European legislation — directives, regulations, decisions. Like "Richtlijn 2011/95/EU" or "Verordening (EU) nr. 1215/2012".
- OEP: References to official publications — Staatsblad (Stb.), Staatscourant (Stcrt.), Kamerstukken.

IMPORTANT RULES:
- Be recall-oriented: when in doubt, include the span. It is better to annotate too much than to miss a reference.
- Copy spans EXACTLY as they appear in the text — character for character, including punctuation and spacing.
- A single article reference including the law name is ONE span: "art. 6:162 BW" is one BWB span.
- Multiple references like "art. 6 en 7 EVRM" are TWO spans: "art. 6 EVRM" and "art. 7 EVRM".
- Also annotate local aliases defined earlier in the text.
- Annotate abbreviations like "Awb", "BW", "Sr" as BWB even without an article number.
- Do NOT annotate: party names, internal references like "zie overweging 3.2", general terms like "de wet" without specifics.

Return ONLY a JSON array. Each item has exactly two fields:
- "span": the exact text copied from the judgment
- "label": one of BWB, ECLI, CELEX, OEP

Example output:
[
  {"span": "artikel 7:671b lid 1, onderdeel a, van het Burgerlijk Wetboek", "label": "BWB"},
  {"span": "ECLI:NL:HR:2023:847", "label": "ECLI"}
]

If no legal references are found, return an empty array: []"""

USER_PROMPT_TEMPLATE = "Annotate all legal references in the following Dutch court judgment:\n\n{text}"

# ── Hulpfuncties ──────────────────────────────────────────────────────────────

def zoek_offsets(tekst, annotations):
    """Zoekt offsets van spans in de tekst."""
    resultaat = []
    zoekpositie = 0

    for ann in annotations:
        if not isinstance(ann, dict) or "span" not in ann or "label" not in ann:
            continue

        span = ann["span"]

        pos = tekst.find(span, zoekpositie)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            zoekpositie = pos + 1
            continue

        # Fallback zonder zoekpositie restrictie
        pos = tekst.find(span)
        if pos != -1:
            resultaat.append({"span": span, "label": ann["label"], "start": pos, "end": pos + len(span)})
            continue

        print(f"  ⚠ Span niet gevonden: '{span}'")

    return resultaat


def chunk_tekst(tekst, max_chars=24000):
    """Splits tekst in stukken op alinea-grenzen."""
    if len(tekst) <= max_chars:
        return [tekst]

    paragraphs = tekst.split("\n\n")
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) > max_chars:
            if current:
                chunks.append(current.strip())
            current = para
        else:
            current += "\n\n" + para
    if current:
        chunks.append(current.strip())
    return chunks


def annotate_judgment(tekst):
    """Annoteert een uitspraak via Claude Haiku."""
    chunks = chunk_tekst(tekst)
    all_annotations = []
    offset = 0

    for chunk in chunks:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": USER_PROMPT_TEMPLATE.format(text=chunk)
            }]
        )

        raw = response.content[0].text.strip()

        # Verwijder markdown als het model die toch toevoegt
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        try:
            parsed = json.loads(raw)
            if isinstance(parsed, list):
                annotations = parsed
            elif isinstance(parsed, dict):
                annotations = next((v for v in parsed.values() if isinstance(v, list)), [])
            else:
                annotations = []

            annotations_met_offsets = zoek_offsets(chunk, annotations)
            for ann in annotations_met_offsets:
                ann["start"] += offset
                ann["end"] += offset
            all_annotations.extend(annotations_met_offsets)

        except json.JSONDecodeError as e:
            print(f"  Parse error op offset {offset}: {e}")

        offset += len(chunk) + 2
        time.sleep(0.3)

    return all_annotations


# ── Hoofdloop ─────────────────────────────────────────────────────────────────

OUTPUT_FILE = "gold_standard_annotations.json"

# Laad al geannoteerde ECLIs zodat je kunt hervatten na onderbreking
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        results = json.load(f)
    al_gedaan = {r["ecli"] for r in results}
    print(f"Hervat: {len(al_gedaan)} uitspraken al gedaan")
else:
    results = []
    al_gedaan = set()

failed = []

for i, row in df_sample.head(10).iterrows():
    if row["ecli"] in al_gedaan:
        continue  # Sla al geannoteerde uitspraken over

    try:
        annotations = annotate_judgment(row["text"])
        results.append({
            "ecli": row["ecli"],
            "annotations": annotations,
            "n_annotations": len(annotations)
        })
        al_gedaan.add(row["ecli"])

        if len(al_gedaan) % 10 == 0:
            print(f"{len(al_gedaan)}/{len(df_sample)} — laatste: {len(annotations)} annotaties")

        # Sla tussentijds op na elke 10 uitspraken
        if len(al_gedaan) % 10 == 0:
            with open(OUTPUT_FILE, "w") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

        time.sleep(0.5)

    except Exception as e:
        print(f"Fout bij {row['ecli']}: {e}")
        failed.append(row["ecli"])

# Sla definitief op
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nKlaar. {len(results)} uitspraken geannoteerd, {len(failed)} mislukt.")
if failed:
    print(f"Mislukt: {failed}")

  ⚠ Span niet gevonden: 'artikel 10, eerste lid, van de Opvangrichtlijn'
10/1000 — laatste: 2 annotaties

Klaar. 10 uitspraken geannoteerd, 0 mislukt.


# Annotaties checken

In [ ]:
from IPython.display import HTML

def toon_annotaties(ecli, results, df_sample):
    """Toont een uitspraak met annotaties gekleurd in de notebook."""
    result = next(r for r in results if r["ecli"] == ecli)
    tekst = df_sample[df_sample["ecli"] == ecli]["text"].iloc[0]
    
    kleuren = {
        "BWB": "#ffeb3b",    # geel
        "ECLI": "#90caf9",   # blauw
        "CELEX": "#a5d6a7",  # groen
        "OEP": "#ffcc80"     # oranje
    }
    
    # Sorteer annotaties op startpositie, langste eerst bij overlap
    anns = sorted(result["annotations"], key=lambda x: (x["start"], -x["end"]))
    
    html = "<style>mark { padding: 2px; border-radius: 3px; }</style>"
    html += "<p style='font-family: Arial; line-height: 1.8;'>"
    
    positie = 0
    for ann in anns:
        if ann["start"] < positie:
            continue  # Sla overlappende spans over
        html += tekst[positie:ann["start"]]
        kleur = kleuren.get(ann["label"], "#e0e0e0")
        html += f'<mark style="background:{kleur}" title="{ann["label"]}">{tekst[ann["start"]:ann["end"]]}</mark>'
        positie = ann["end"]
    
    html += tekst[positie:] + "</p>"
    
    legenda = " ".join([f'<span style="background:{k}; padding:3px 6px; border-radius:3px">{v}</span>' 
                        for v, k in kleuren.items()])
    
    return HTML(f"<b>{ecli}</b> — {result['n_annotations']} annotaties<br>{legenda}<br><br>{html}")

In [ ]:
with open("gold_standard_annotations.json") as f:
    results = json.load(f)

eclis = [r["ecli"] for r in results]
print(f"Totaal: {len(eclis)} uitspraken")
print(eclis)

Totaal: 10 uitspraken
['ECLI:NL:RBDHA:2025:8235', 'ECLI:NL:OGAACMB:2017:68', 'ECLI:NL:RBSGR:2012:BW2477', 'ECLI:NL:RBDHA:2021:16863', 'ECLI:NL:RBDHA:2025:4857', 'ECLI:NL:RBNNE:2024:4711', 'ECLI:NL:RVS:2025:2901', 'ECLI:NL:RBARN:2006:3841', 'ECLI:NL:RBDHA:2025:13255', 'ECLI:NL:CBB:2025:206']


In [ ]:
for ecli in eclis:
    display(toon_annotaties(ecli, results, df_sample))